# Clinical info plotting script.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [ ]:
df = pd.read_csv(f'{ACC_DATA_ROOT}/clinical_info.csv')

In [ ]:
df

In [ ]:
# configuring output locations
# ---------- CONFIG ----------
INPUT_CSV = f"{ACC_DATA_ROOT}/clinical_info.csv"
OUTDIR = Path(f"{ACC_DATA_ROOT}/bulk_rna_data/outputs/images/clinical_outcome")
# OUTDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# defining custome color palettes
# Color palettes
# BINARY_PALETTE = {
#     "Poor": "#E63946",  # red
#     "Good": "#2A9D8F",  # green
# }

BINARY_PALETTE = {
    "Poor": "#E63946",  # red
    "Good": "#2c9d2a",  # green
}

THREE_LEVEL_PALETTE = {
    "< 2 years": "#E63946",  # red
    "~ 3 years": "#F4A261",  # amber
    "> 5 years": "#2c9d2a",  # green
}

# Site colors (you can customize these)
# SITE_PALETTE = {
#     "Base of Tongue": "#1f77b4",
#     "Maxilla": "#ff7f0e",
#     "Tongue": "#2ca02c",
#     "Submandibular Gland": "#d62728",
#     "Parotid Gland": "#9467bd",
#     "Unknown": "#8c564b",
#     "Larynx": "#e377c2",
#     "Floor of Mouth": "#7f7f7f",
#     "Mandible": "#bcbd22",
#     "Maxillary Sinus": "#17becf",
#     "Buccal Mucosa": "#aec7e8",
# }

SITE_PALETTE = {
    "Base of Tongue": "#0aa8f7",
    "Maxilla": "#0aa8f7",
    "Tongue": "#0aa8f7",
    "Submandibular Gland": "#0aa8f7",
    "Parotid Gland": "#0aa8f7",
    "Unknown": "#0aa8f7",
    "Larynx": "#0aa8f7",
    "Floor of Mouth": "#0aa8f7",
    "Mandible": "#0aa8f7",
    "Maxillary Sinus": "#0aa8f7",
    "Buccal Mucosa": "#0aa8f7",
}

# Category colors for stacking
CATEGORY_PALETTE = {
    "Poor": "#E63946",
    "Good": "#2c9d2a",
}

In [ ]:
def load_data(path):
    """Load and prepare the clinical data"""
    df = pd.read_csv(path)
    
    # Map clinical outcome (numeric) to 3-level categories
    def map_outcome_to_3level(value):
        try:
            val = float(value)
            if val < 2.5:
                return "< 2 years"
            elif val < 4:
                return "~ 3 years"
            else:
                return "> 5 years"
        except:
            return np.nan
    
    df['Outcome_3Level'] = df['Years'].apply(map_outcome_to_3level)
    
    return df

In [ ]:
# plotting style
def style_matplotlib():
    plt.rcParams.update({
        "figure.dpi": 300,
        "savefig.dpi": 600,
        "font.family": "sans-serif",
        # 'Arial Narrow' is proprietary and is NOT installed on this machine;
        # 'Liberation Sans Narrow' is its metric-compatible substitute and is what
        # actually renders here. Arial Narrow is listed first so any machine that does
        # have it picks up the real font.
        "font.sans-serif": ['Arial Narrow', 'Liberation Sans Narrow',
                            'Nimbus Sans Narrow', 'DejaVu Sans'],
        "font.size": 9,
        "axes.titlesize": 9,
        "axes.labelsize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "legend.title_fontsize": 9,
        # keep text as text in the SVG (editable in Illustrator, resolves the real font
        # wherever it is installed) instead of matplotlib's default outlined paths
        "svg.fonttype": "none",
        "axes.spines.top": False,
        "axes.spines.right": False
    })

In [ ]:
# ---------- PLOT 1: Binary Outcome Distribution ----------
def plot_binary_outcome(df, outbase="outcome_distribution_binary_bar_plot_clinical"):
    """Plot Poor vs Good outcome distribution"""
    style_matplotlib()
    
    # Count Poor/Good
    counts = df['Clinical outcome'].value_counts()
    # counts = df['Model outcome'].value_counts()
    total = len(df)
    pct = (counts / total * 100).round(1)
    
    # Ensure order
    categories = ['Good', 'Poor']
    counts = counts.reindex(categories, fill_value=0)
    pct = pct.reindex(categories, fill_value=0)
    
    colors = [BINARY_PALETTE[c] for c in categories]
    
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(categories, counts, color=colors, width=0.6)
    
    # Annotate N (p%)
    for rect, cnt, p in zip(bars, counts, pct):
        ax.annotate(
            f"{cnt} ({p:.0f}%)",
            xy=(rect.get_x() + rect.get_width() / 2, rect.get_height()),
            xytext=(0, 6),
            textcoords="offset points",
            ha="center",
            va="bottom",
        )
    
    ax.set_ylabel("Number of Patients")
    ax.set_xlabel("Clinical Outcome")
    ax.set_title(f"Clinical Outcome Distribution (N={total})", pad=20, weight = 'bold', fontsize = 14)
    # ax.set_title(f"Model Outcome Distribution (N={total})", pad=20, weight = 'bold', fontsize = 14)
    ax.set_ylim(0, counts.max() * 1.25)
    
    fig.tight_layout()
    # fig.savefig(OUTDIR / f"{outbase}.pdf", bbox_inches="tight")
    # fig.savefig(OUTDIR / f"{outbase}.tiff", bbox_inches="tight")
    fig.savefig(OUTDIR / f"{outbase}.svg", 
                bbox_inches="tight", 
                format = 'svg')
    plt.close(fig)
    print(f"✓ Saved {outbase}")

In [ ]:
plot_binary_outcome(df)

In [ ]:
# ---------- PLOT 1: Binary Outcome Distribution (Pie Chart) ----------
def plot_binary_outcome_pie(df, outbase="outcome_distribution_binary_pie_clinical"):
    """Plot Poor vs Good outcome distribution as a pie chart"""
    style_matplotlib()

    # Count Poor/Good
    counts = df['Clinical outcome'].value_counts()
    # counts = df['Model outcome'].value_counts()
    total = len(df)
    pct = (counts / total * 100).round(1)

    # Ensure order
    categories = ['Good', 'Poor']
    counts = counts.reindex(categories, fill_value=0)
    pct = pct.reindex(categories, fill_value=0)

    colors = [BINARY_PALETTE[c] for c in categories]

    fig, ax = plt.subplots(figsize=(6, 4))

    # Create pie chart
    wedges, texts, autotexts = ax.pie(
        counts,
        labels=categories,
        colors=colors,
        autopct='%1.1f%%',
        startangle=90,
        explode=(0.005, 0.005),  # Slight separation for visual clarity
        textprops={'fontsize': 12, 'weight': 'bold'}
    )

    # Customize percentage text color to white for better visibility
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(11)

    # Add legend with count information
    legend_labels = [f"{cat}: n={int(cnt)} ({p:.1f}%)"
                     for cat, cnt, p in zip(categories, counts, pct)]
    ax.legend(legend_labels, loc="upper left", bbox_to_anchor=(1, 0, 0.5, 1))

    ax.set_title(f"Clinical Outcome Distribution (N={total})", pad=20, fontsize=14, weight='bold')
    # ax.set_title(f"Model Outcome Distribution (N={total})", pad=20, fontsize=14, weight='bold')

    fig.tight_layout()
    # fig.savefig(OUTDIR / f"{outbase}.pdf", bbox_inches="tight")
    # fig.savefig(OUTDIR / f"{outbase}.tiff", bbox_inches="tight")
    fig.savefig(OUTDIR / f"{outbase}.svg", 
                bbox_inches="tight",
                format = 'svg')
    plt.close(fig)
    print(f"✓ Saved {outbase}")

In [ ]:
plot_binary_outcome_pie(df)

In [ ]:
# # ---------- PLOT 2: 3-Level Outcome Distribution ----------
# def plot_3level_outcome(df, outbase="outcome_distribution_3level"):
#     """Plot <2, ~3, >5 years outcome distribution"""
#     style_matplotlib()
    
#     # Count outcomes
#     category_order = ["< 2 years", "~ 3 years", "> 5 years"]
#     counts = df['Outcome_3Level'].value_counts().reindex(category_order, fill_value=0)
#     total = counts.sum()
#     pct = (counts / total * 100).round(1)
    
#     colors = [THREE_LEVEL_PALETTE[c] for c in category_order]
    
#     fig, ax = plt.subplots(figsize=(5, 4.0))
#     bars = ax.bar(category_order, counts, color=colors, width=0.7)
    
#     # Annotate N (p%)
#     for rect, cnt, p in zip(bars, counts, pct):
#         ax.annotate(
#             f"{int(cnt)} ({p:.0f}%)",
#             xy=(rect.get_x() + rect.get_width() / 2, rect.get_height()),
#             xytext=(0, 6),
#             textcoords="offset points",
#             ha="center",
#             va="bottom",
#         )
    
#     ax.set_ylabel("Number of Patients")
#     ax.set_xlabel("Survival Time")
#     ax.set_title(f"Outcome Distribution (N={int(total)})", pad=10)
#     ax.set_ylim(0, counts.max() * 1.25)
#     plt.xticks(rotation=0)
    
#     fig.tight_layout()
#     fig.savefig(OUTDIR / f"{outbase}.pdf", bbox_inches="tight")
#     fig.savefig(OUTDIR / f"{outbase}.tiff", bbox_inches="tight")
#     plt.close(fig)
#     print(f"✓ Saved {outbase}")

In [ ]:
# ---------- PLOT 3: Site Composition ----------
def plot_site_composition(df, outbase="site_composition"):
    """Plot distribution across anatomic sites"""
    style_matplotlib()
    
    # Count by site
    site_counts = df['Site of Extraction'].value_counts().sort_values(ascending=False)
    total = len(df)
    pct = (site_counts / total * 100).round(1)
    
    # Get colors
    colors = [SITE_PALETTE.get(site, "#999999") for site in site_counts.index]
    
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.barh(range(len(site_counts)), site_counts.values, color=colors)
    
    # Annotate N (p%)
    for i, (cnt, p) in enumerate(zip(site_counts.values, pct.values)):
        ax.annotate(
            f"{cnt} ({p:.0f}%)",
            xy=(cnt, i),
            xytext=(6, 0),
            textcoords="offset points",
            ha="left",
            va="center",
        )
    
    ax.set_yticks(range(len(site_counts)))
    ax.set_yticklabels(site_counts.index)
    ax.set_xlabel("Number of Patients")
    ax.set_title(f"Site of Extraction (N={total})", pad=20, fontsize = 14, weight = 'bold')
    ax.set_xlim(0, site_counts.max() * 1.35)
    ax.invert_yaxis()
    
    fig.tight_layout()
    # fig.savefig(OUTDIR / f"{outbase}.pdf", bbox_inches="tight")
    # fig.savefig(OUTDIR / f"{outbase}.tiff", bbox_inches="tight")
    fig.savefig(OUTDIR / f"{outbase}.svg", 
                bbox_inches="tight",
                format = 'svg')
    plt.close(fig)
    print(f"✓ Saved {outbase}")

In [ ]:
# ---------- PLOT 3: Site Composition ----------
def plot_site_composition(df, outbase="site_composition"):
    """Plot distribution across anatomic sites"""
    style_matplotlib()
    
    # Count by site
    site_counts = df['Site of Extraction'].value_counts().sort_values(ascending=False)
    total = len(df)
    # pct = (site_counts / total * 100).round(1)
    
    # Get colors
    colors = [SITE_PALETTE.get(site, "#999999") for site in site_counts.index]
    
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.barh(range(len(site_counts)), site_counts.values, color=colors)
    
    # Annotate N (p%)
    # for i, (cnt, p) in enumerate(zip(site_counts.values, pct.values)):
    # for i, cnt in enumerate(site_counts.values):
    # ax.annotate(
    #     f"{cnt}",
    #     xy=(cnt, i),
    #     xytext=(6, 0),
    #     textcoords="offset points",
    #     ha="left",
    #     va="center",
    # )
    
    ax.set_yticks(range(len(site_counts)))
    ax.set_yticklabels(site_counts.index)
    ax.set_xlabel("Number of Patients")
    ax.set_title(f"Site of Extraction (N={total})", pad=20, fontsize = 14, weight = 'bold')
    ax.set_xlim(0, site_counts.max() * 1.35)
    ax.invert_yaxis()
    
    fig.tight_layout()
    # fig.savefig(OUTDIR / f"{outbase}.pdf", bbox_inches="tight")
    # fig.savefig(OUTDIR / f"{outbase}.tiff", bbox_inches="tight")
    fig.savefig(OUTDIR / f"{outbase}.svg", 
                bbox_inches="tight",
                format = 'svg')
    plt.close(fig)
    print(f"✓ Saved {outbase}")

In [ ]:
plot_site_composition(df)

In [ ]:
# ---------- PLOT 4: Site Composition Stacked by Clinical Category ----------
def plot_site_by_category(df, outbase="site_composition_stacked_clinical"):
    """Plot sites stacked by Poor/Good clinical category"""
    style_matplotlib()
    
    # The CSV collapses all four base-of-tongue samples into one 'Base of Tongue' value,
    # but the source h5ad keeps the raw distinction: P09/P11 are 'BOT' while
    # P15/P20 are 'Baseoftongue'. These are to be plotted as separate sites.
    BOT_SAMPLE_IDS = {'P09', 'P11'}
    df = df.copy()
    bot_rows = df['Sample ID'].isin(BOT_SAMPLE_IDS)
    assert bot_rows.sum() == len(BOT_SAMPLE_IDS), \
        "BOT sample IDs missing from the CSV - check 'Sample ID' values"
    assert (df.loc[bot_rows, 'Site of Extraction'] == 'Base of Tongue').all(), \
        "A BOT sample is no longer labelled 'Base of Tongue' in the CSV"
    df.loc[bot_rows, 'Site of Extraction'] = 'BOT'

    # Create cross-tabulation
    crosstab = pd.crosstab(df['Site of Extraction'], df['Clinical outcome'])
    # crosstab = pd.crosstab(df['Site of Extraction'], df['Model outcome'])
    
    # Ensure Poor/Good columns exist
    for cat in ['Good', 'Poor']:
        if cat not in crosstab.columns:
            crosstab[cat] = 0
    
    # Sort by total count
    crosstab['Total'] = crosstab['Good'] + crosstab['Poor']
    crosstab = crosstab.sort_values('Total', ascending=False).drop('Total', axis=1)
    
    # Calculate percentages for annotations
    row_totals = crosstab.sum(axis=1)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # Create stacked bars
    bottom = np.zeros(len(crosstab))
    for category in ['Good', 'Poor']:
        if category in crosstab.columns:
            bars = ax.barh(
                range(len(crosstab)),
                crosstab[category],
                left=bottom,
                label=category,
                color=CATEGORY_PALETTE[category]
            )
            
            # Annotate each segment with count if > 0
            for i, (val, bot) in enumerate(zip(crosstab[category], bottom)):
                if val > 0:
                    ax.text(
                        bot + val/2,
                        i,
                        str(int(val)),
                        ha='center',
                        va='center',
                        color='white',
                        fontweight='bold',
                        fontsize=9
                    )
            
            bottom += crosstab[category]
    
    # # Add total counts at the end of bars
    # for i, total in enumerate(row_totals):
    #     ax.text(
    #         total + 0.5,
    #         i,
    #         f"n={int(total)}",
    #         ha='left',
    #         va='center',
    #         fontsize=9
    #     )
    
    ax.set_yticks(range(len(crosstab)))
    ax.set_yticklabels(crosstab.index)
    ax.set_xlabel("Number of Patients")
    ax.set_title(f"Site Composition by Clinical Category (N={len(df)})", pad=20, fontsize = 9, weight = 'bold')
    ax.set_xlim(0, row_totals.max() * 1.25)
    ax.legend(title='Clinical Category', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.invert_yaxis()
    
    fig.tight_layout()
    # fig.savefig(OUTDIR / f"{outbase}.pdf", bbox_inches="tight")
    # fig.savefig(OUTDIR / f"{outbase}.tiff", bbox_inches="tight")
    fig.savefig(OUTDIR / f"{outbase}.svg", 
                bbox_inches="tight",
                format = 'svg')
    plt.close(fig)
    print(f"✓ Saved {outbase}")

In [ ]:
plot_site_by_category(df)

In [ ]:
# ---------- PLOT 5: Sample Overlap Across Data Modalities ----------
def plot_modality_venn(df, outbase="modality_overlap_venn"):
    """Plot overlap of the Bulk RNA / Single Cell RNA / Spatial sample sets"""
    style_matplotlib()
    
    from matplotlib_venn import venn3, venn3_circles
    from matplotlib_venn.layout.venn3 import DefaultLayoutAlgorithm
    
    # Spatial is an object column mixing 0 with 'Visium' / 'Visium HD'
    spatial_flag = df['Spatial Datasets'].astype(str).str.strip().ne('0')
    bulk = set(df.loc[df['Bulk RNA Datasets'] == 1, 'Sample ID'])
    sc = set(df.loc[df['Single Cell RNA Datasets'] == 1, 'Sample ID'])
    spatial = set(df.loc[spatial_flag, 'Sample ID'])
    
    regions = {
        '100': len(bulk - sc - spatial),
        '010': len(sc - bulk - spatial),
        '110': len((bulk & sc) - spatial),
        '001': len(spatial - bulk - sc),
        '101': len((bulk & spatial) - sc),
        '011': len((sc & spatial) - bulk),
        '111': len(bulk & sc & spatial),
    }
    
    # Fail loudly if the CSV ever changes rather than drawing a wrong figure
    assert sum(regions.values()) == len(df), "Venn regions do not sum to N"
    assert len(bulk) == int((df['Bulk RNA Datasets'] == 1).sum())
    assert len(sc) == int((df['Single Cell RNA Datasets'] == 1).sum())
    assert len(spatial) == int(spatial_flag.sum())
    
    set_labels = (f"Bulk RNA\n(n={len(bulk)})",
                  f"Single Cell RNA\n(n={len(sc)})",
                  f"Spatial\n(n={len(spatial)})")
    set_colors = ("#0aa8f7", "#9467bd", "#F4A261")
    
    # Equal-size circles: geometry carries no quantity, every region is labelled instead
    layout = DefaultLayoutAlgorithm(fixed_subset_sizes=(1,) * 7)
    
    fig, ax = plt.subplots(figsize=(7, 6))
    v = venn3([bulk, sc, spatial], set_labels=set_labels, set_colors=set_colors,
              alpha=0.55, ax=ax, layout_algorithm=layout)
    venn3_circles([bulk, sc, spatial], linewidth=1, ax=ax, layout_algorithm=layout)
    
    # Label all 7 regions, including the empty ones (in gray)
    for region, n in regions.items():
        label = v.get_label_by_id(region)
        label.set_text(str(n))
        if n > 0:
            label.set_fontsize(13)
            label.set_fontweight('bold')
        else:
            label.set_fontsize(10)
            label.set_color('#9e9e9e')
    
    for label in v.set_labels:
        label.set_fontsize(12)
        label.set_fontweight('bold')
    
    ax.set_title(f"Sample Overlap Across Data Modalities (N={len(df)})", pad=20, fontsize = 14, weight = 'bold')
    
    n_visium = int(df['Spatial Datasets'].astype(str).str.strip().eq('Visium').sum())
    n_visium_hd = int(df['Spatial Datasets'].astype(str).str.strip().eq('Visium HD').sum())
    
    fig.tight_layout()
    fig.text(0.5, -0.02,
             f"All {len(spatial)} spatial samples are also bulk- and single-cell-profiled "
             f"({n_visium} Visium, {n_visium_hd} Visium HD).",
             ha='center', fontsize=9, color='#555555')
    # fig.savefig(OUTDIR / f"{outbase}.pdf", bbox_inches="tight")
    # fig.savefig(OUTDIR / f"{outbase}.tiff", bbox_inches="tight")
    fig.savefig(OUTDIR / f"{outbase}.svg", 
                bbox_inches="tight",
                format = 'svg')
    plt.close(fig)
    print(f"✓ Saved {outbase}")
    
    print(f"\nSet totals: Bulk={len(bulk)}, Single Cell={len(sc)}, Spatial={len(spatial)}, N={len(df)}")
    print("Region counts:")
    for region, name in [('111', 'Bulk ∩ SC ∩ Spatial'), ('110', 'Bulk ∩ SC only'),
                         ('100', 'Bulk only'), ('010', 'Single Cell only'),
                         ('001', 'Spatial only'), ('101', 'Bulk ∩ Spatial only'),
                         ('011', 'SC ∩ Spatial only')]:
        print(f"  {name:22s} {regions[region]}")

In [ ]:
plot_modality_venn(df)

In [ ]:
# ---------- PLOT 6: Assay Availability Across the Cohort ----------
def plot_modality_availability(df, outbase="modality_availability_grid"):
    """Plot which assays were run on each patient, sorted by modality profile"""
    style_matplotlib()
    
    from matplotlib.patches import Rectangle
    
    # Spatial is an object column mixing 0 with 'Visium' / 'Visium HD'
    platform = df['Spatial Datasets'].astype(str).str.strip()
    has_bulk = df['Bulk RNA Datasets'] == 1
    has_sc = df['Single Cell RNA Datasets'] == 1
    has_spatial = platform.ne('0')
    
    # This group order keeps every modality row a contiguous run
    groups = [("Bulk only", has_bulk & ~has_sc & ~has_spatial),
              ("All three", has_bulk & has_sc & has_spatial),
              ("Bulk + SC", has_bulk & has_sc & ~has_spatial),
              ("SC only", ~has_bulk & has_sc & ~has_spatial)]
    
    d = df.assign(_platform=platform, _group=np.select([m for _, m in groups],
                                                       [n for n, _ in groups], default="none"))
    d['_order'] = d['_group'].map({n: i for i, (n, _) in enumerate(groups)})
    d = d.sort_values(['_order', 'Sample ID']).reset_index(drop=True)
    
    sizes = [int(m.sum()) for _, m in groups]
    
    # Fail loudly if the CSV ever changes rather than drawing a wrong figure
    assert sum(sizes) == len(df), "Profile groups do not sum to N"
    assert (d['_group'] != "none").all(), "A sample has no modality at all"
    assert set(d.loc[d['_platform'].ne('0'), 'Sample ID']) <= set(
        df.loc[has_bulk & has_sc, 'Sample ID']), "A spatial sample lacks bulk or single-cell data"
    
    rows = [("Bulk RNA", d['Bulk RNA Datasets'] == 1, "#0aa8f7"),
            ("Single Cell RNA", d['Single Cell RNA Datasets'] == 1, "#9467bd"),
            ("Spatial", d['_platform'].ne('0'), "#F4A261")]
    HD_COLOR = "#C1662F"
    EMPTY = "#ececec"
    
    for name, mask, _ in rows:
        assert int(mask.sum()) == int({"Bulk RNA": has_bulk, "Single Cell RNA": has_sc,
                                       "Spatial": has_spatial}[name].sum()), f"{name} recount failed"
    
    fig = plt.figure(figsize=(11, 4.2))
    gs = fig.add_gridspec(2, 2, height_ratios=[3, 0.5], width_ratios=[6, 1.2],
                          hspace=0.15, wspace=0.04)
    ax_grid = fig.add_subplot(gs[0, 0])
    ax_tot = fig.add_subplot(gs[0, 1])
    ax_out = fig.add_subplot(gs[1, 0], sharex=ax_grid)
    fig.add_subplot(gs[1, 1]).set_axis_off()
    
    # Cells: filled = assayed, with a gap between cells for the oncoprint look
    for r, (name, mask, color) in enumerate(rows):
        for i in range(len(d)):
            if not mask.iloc[i]:
                face = EMPTY
            elif name == "Spatial":
                face = HD_COLOR if d['_platform'].iloc[i] == "Visium HD" else color
            else:
                face = color
            ax_grid.add_patch(Rectangle((i - 0.41, r - 0.41), 0.82, 0.82,
                                        facecolor=face, edgecolor='none'))
    
    # Group separators and labels
    bounds = np.cumsum(sizes)[:-1] - 0.5
    for b in bounds:
        ax_grid.axvline(b, color='#9e9e9e', linewidth=0.9)
        ax_out.axvline(b, color='#9e9e9e', linewidth=0.9)
    
    starts = np.concatenate([[0], np.cumsum(sizes)[:-1]])
    for (name, _), start, size in zip(groups, starts, sizes):
        ax_grid.text(start + (size - 1) / 2, -0.75, f"{name}\n(n={size})",
                     ha='center', va='bottom', fontsize=9)
    
    ax_grid.set_xlim(-0.6, len(d) - 0.4)
    ax_grid.set_ylim(-0.6, len(rows) - 0.4)
    ax_grid.invert_yaxis()
    ax_grid.set_yticks(range(len(rows)))
    ax_grid.set_yticklabels([n for n, _, _ in rows], fontsize=11)
    ax_grid.tick_params(axis='both', length=0)
    plt.setp(ax_grid.get_xticklabels(), visible=False)
    for s in ax_grid.spines.values():
        s.set_visible(False)
    ax_grid.set_title(f"Assay Availability Across the ACC Cohort (N={len(df)})",
                      pad=38, fontsize = 14, weight = 'bold')
    
    # Outcome strip
    for i in range(len(d)):
        ax_out.add_patch(Rectangle((i - 0.41, -0.41), 0.82, 0.82,
                                   facecolor=BINARY_PALETTE[d['Clinical outcome'].iloc[i]],
                                   edgecolor='none'))
    ax_out.set_ylim(-0.6, 0.6)
    ax_out.set_yticks([0])
    ax_out.set_yticklabels(["Outcome"], fontsize=11)
    ax_out.set_xticks(range(len(d)))
    ax_out.set_xticklabels(d['Sample ID'], rotation=90, fontsize=7)
    ax_out.tick_params(axis='both', length=0)
    for s in ax_out.spines.values():
        s.set_visible(False)
    
    # Per-modality totals, with the spatial bar split by platform
    n_visium = int((d['_platform'] == "Visium").sum())
    n_hd = int((d['_platform'] == "Visium HD").sum())
    for r, (name, mask, color) in enumerate(rows):
        total = int(mask.sum())
        if name == "Spatial":
            ax_tot.barh(r, n_visium, height=0.55, color=color)
            ax_tot.barh(r, n_hd, left=n_visium, height=0.55, color=HD_COLOR)
        else:
            ax_tot.barh(r, total, height=0.55, color=color)
        ax_tot.text(total + 0.6, r, str(total), va='center', ha='left',
                    fontsize=10, fontweight='bold')
    
    ax_tot.set_xlim(0, len(df) * 1.18)
    ax_tot.set_ylim(-0.6, len(rows) - 0.4)
    ax_tot.invert_yaxis()
    ax_tot.set_yticks([])
    ax_tot.set_xticks([])
    for s in ax_tot.spines.values():
        s.set_visible(False)
    ax_tot.set_title("Total", fontsize=10, pad=6)
    
    handles = [Rectangle((0, 0), 1, 1, facecolor=c) for c in
               ["#F4A261", HD_COLOR, EMPTY, BINARY_PALETTE['Good'], BINARY_PALETTE['Poor']]]
    labels = ["Visium", "Visium HD", "Not assayed", "Good outcome", "Poor outcome"]
    fig.legend(handles, labels, loc='lower center', ncol=5, frameon=False,
               fontsize=9, bbox_to_anchor=(0.5, -0.13))
    
    fig.tight_layout()
    # fig.savefig(OUTDIR / f"{outbase}.pdf", bbox_inches="tight")
    # fig.savefig(OUTDIR / f"{outbase}.tiff", bbox_inches="tight")
    fig.savefig(OUTDIR / f"{outbase}.svg", 
                bbox_inches="tight",
                format = 'svg')
    plt.close(fig)
    print(f"✓ Saved {outbase}")
    
    print(f"\nSet totals: Bulk={int(has_bulk.sum())}, Single Cell={int(has_sc.sum())}, "
          f"Spatial={int(has_spatial.sum())} ({n_visium} Visium, {n_hd} Visium HD), N={len(df)}")
    print("Profile groups:")
    for (name, _), size in zip(groups, sizes):
        print(f"  {name:12s} {size}")

In [ ]:
plot_modality_availability(df)

In [ ]:
# # ---------- MAIN ----------
# def main():
#     # Load data
#     df = load_data(INPUT_CSV)
    
#     print(f"\nDataset Overview:")
#     print(f"Total samples: {len(df)}")
#     print(f"\nClinical outcome:\n{df['Clinical outcome'].value_counts()}")
#     print(f"\n3-Level Outcomes:\n{df['Outcome_3Level'].value_counts()}")
#     print(f"\nSites:\n{df['Site of Extraction'].value_counts()}")
    
#     print("\n" + "="*50)
#     print("Generating Cohort Overview Figures...")
#     print("="*50)
    
#     # Generate all plots
#     plot_binary_outcome(df)
#     plot_3level_outcome(df)
#     plot_site_composition(df)
#     plot_site_by_category(df)
    
#     print(f"\n✓ All figures saved to {OUTDIR}/")
#     print("\nGenerated files:")
#     print("  - outcome_distribution_binary.pdf/.tiff")
#     print("  - outcome_distribution_3level.pdf/.tiff")
#     print("  - site_composition.pdf/.tiff")
#     print("  - site_composition_stacked.pdf/.tiff")

# if __name__ == "__main__":
#     main()